[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/071.%20The%20Classic%20Vehicle%20Routing%20Problem%20%28VRP%29/P71-Tier-7_executed.ipynb)

---


# 71. The Classic Vehicle Routing Problem (VRP)

## Tier 7: The Human–AI Symbiotic Partnership (The Centaur Model)

### Purpose

This notebook demonstrates a human–AI collaboration framework for a simplified vehicle-routing problem. Human experts and an AI system independently produce route recommendations. A collaboration step compares their recommendations, may adopt a substantially lower-cost human recommendation, and then applies route-level local search. The simulation also tracks confidence, route-cost agreement, and a rule-based trust measure.

The goal is to understand how the partnership is represented and evaluated in code—not to claim that the simulation establishes real-world decision accuracy, trust, or business impact.

### Inputs and outputs

The inputs are generated customer coordinates, a depot, customer demands, vehicle records, and simulated human-expert and AI-system attributes. The outputs are route recommendations, Euclidean route costs, confidence and trust metrics, and plots summarizing the simulated decisions.

### Scope and modeling assumptions

- The route-cost measure is the sum of straight-line Euclidean distances from the depot through each route and back to the depot. Distances are in the same unspecified coordinate units as the generated locations.
- Customer demands, vehicle capacities, time windows, traffic, service times, and priorities are generated or stored as problem data, but they are **not enforced** by the route-generation methods. Consequently, the produced routes are complete partitions of the customers, but are not guaranteed to be feasible for the full capacitated VRP with time windows.
- The human and AI recommendations are simulated heuristics, not observed decisions or a trained optimization model. The confidence values are assigned model attributes; they are not calibrated probabilities or measured accuracy.
- The trust update is a small, explicit rule based on the cost of the collaborative route relative to the average human route cost. Repeating decisions does not train the AI or incorporate feedback into its routing policy.
- Randomized elements use fixed NumPy and Python random seeds for reproducibility. Decision timestamps and measured execution times depend on the runtime environment.

### Method and mathematical representation

Let the customer set be $C=\{1,\ldots,n\}$, let $0$ denote the depot, and let $R=(R_1,\ldots,R_m)$ be the routes for $m$ vehicles. Each recommendation assigns every customer to exactly one route. For a customer $i$ at location $(x_i,y_i)$ and a depot at $(x_0,y_0)$, the distance between locations $i$ and $j$ is

$$
d_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}.
$$

The cost used by the code is

$$
J(R)=\sum_{k=1}^{m}\left(d_{0,r_{k,1}}+
\sum_{t=1}^{|R_k|-1}d_{r_{k,t},r_{k,t+1}}+
d_{r_{k,|R_k|},0}\right),
$$

where $R_k=(r_{k,1},\ldots,r_{k,|R_k|})$ is vehicle $k$'s ordered customer sequence. An empty route contributes zero. In the implementation, `HumanAISymbioticSystem._calculate_route_cost` computes this objective.

The decision sequence is: generate human recommendations $H_1,\ldots,H_q$ and an AI recommendation $A$; set the collaborative starting route to $A$; if the least-cost human recommendation has cost below $0.9J(A)$, use that complete human recommendation instead; then apply 2-opt local search to each route. In symbols, the selection step is

$$
R^{(0)}=\begin{cases}
\displaystyle\arg\min_{H_j}J(H_j), & \text{if }\displaystyle\min_j J(H_j)<0.9J(A),\\
A, & \text{otherwise},
\end{cases}
$$

followed by route-level local improvement to obtain $R^{\mathrm{collab}}$. The 2-opt step examines reversals of contiguous route segments and accepts a reversal only when it reduces that route's depot-to-depot distance. It stops when a pass finds no improving reversal or when the pass limit is reached. This is a local heuristic, not a proof of global optimality.

For $n$ customers, the distance objective can be evaluated in $O(n)$ time for a fixed set of routes. A full pass through all segment reversals of a route of length $s$ examines $O(s^2)$ candidates, each evaluated in $O(s)$ time here, giving $O(s^3)$ work per pass. The number of passes is capped in the implementation. The notebook does not enumerate all possible VRP assignments, whose number grows rapidly with the number of customers.

### Tier 7 partnership concepts

1. **Human expertise:** domain knowledge is represented by expert profiles and simulated recommendation rules.
2. **AI recommendation:** a clustering-style assignment and route-ordering heuristic produces a separate recommendation.
3. **Transparent comparison:** route costs and short reasoning summaries are recorded for each recommendation.
4. **Collaborative refinement:** the system selects between complete recommendations and performs local route improvement.
5. **Trust monitoring:** a specified update rule records changes in simulated mutual trust.
6. **Performance monitoring:** costs, confidence measures, agreement, and elapsed time are summarized without treating them as empirical accuracy or causal evidence.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds make the randomized instance and heuristic choices reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class HumanExpert:
    """Represent a simulated logistics expert and their domain profile."""

    expert_id: int
    name: str
    experience_years: int
    specializations: List[str]
    confidence_level: float
    decision_history: List[Dict] = field(default_factory=list)
    trust_in_ai: float = 0.5
    feedback_history: List[Dict] = field(default_factory=list)


@dataclass
class AIOptimizer:
    """Represent the simulated AI routing recommender."""

    system_id: int
    model_version: str
    optimization_capabilities: List[str]
    confidence_level: float
    decision_history: List[Dict] = field(default_factory=list)
    learning_rate: float = 0.1
    explainability_score: float = 0.8


@dataclass
class VRPDecision:
    """Store recommendations and measured attributes for one decision."""

    decision_id: int
    timestamp: datetime
    problem_instance: Dict
    human_recommendation: Optional[List[List[int]]]
    ai_recommendation: Optional[List[List[int]]]
    collaborative_decision: Optional[List[List[int]]]
    human_confidence: float
    ai_confidence: float
    final_confidence: float
    outcome: Dict


@dataclass
class TrustMetrics:
    """Track the simulation's human–AI trust state."""

    human_trust_in_ai: float
    ai_trust_in_human: float
    mutual_trust: float
    trust_history: List[Tuple[datetime, float]] = field(default_factory=list)
    trust_factors: Dict[str, float] = field(default_factory=dict)


@dataclass
class LearningFeedback:
    """Describe a feedback record; this notebook does not apply it to the AI."""

    feedback_id: int
    expert_id: int
    decision_id: int
    feedback_type: str
    original_ai_decision: List[List[int]]
    human_correction: List[List[int]]
    reasoning: str
    impact_score: float
    timestamp: datetime

In [ ]:
class HumanAISymbioticSystem:
    """Simulate human–AI recommendations and collaborative VRP refinement."""

    def __init__(self, num_experts: int = 5, ai_system_version: str = "v2.1"):
        if num_experts < 1:
            raise ValueError("At least one human expert is required.")

        self.num_experts = num_experts
        self.ai_system_version = ai_system_version
        self.human_experts = self._initialize_human_experts()
        self.ai_optimizer = self._initialize_ai_system()

        self.trust_metrics = TrustMetrics(
            human_trust_in_ai=0.5,
            ai_trust_in_human=0.5,
            mutual_trust=0.5,
            trust_factors={
                "accuracy": 0.0,
                "explainability": self.ai_optimizer.explainability_score,
                "reliability": 0.0,
                "responsiveness": 0.0,
            },
        )
        self.decision_history: List[VRPDecision] = []
        self.learning_feedback: List[LearningFeedback] = []
        self.performance_metrics = {
            "human_only_cost": [],
            "ai_only_cost": [],
            "collaborative_cost": [],
            "decision_time": [],
            "learning_effectiveness": [],
            "trust_development": [],
            "partnership_satisfaction": [],
        }
        self.partnership_stage = "forming"
        self.collaboration_count = 0
        self.conflict_count = 0
        self.learning_iterations = 0

    def _initialize_human_experts(self) -> List[HumanExpert]:
        """Create experts with varied specializations and simulated confidence."""
        specializations = [
            ["urban_routing", "traffic_patterns"],
            ["suburban_logistics", "capacity_planning"],
            ["cost_optimization", "fuel_efficiency"],
            ["customer_service", "time_windows"],
            ["risk_management", "contingency_planning"],
        ]
        names = [
            "Dr. Sarah Chen",
            "Michael Rodriguez",
            "Emily Watson",
            "David Kim",
            "Lisa Anderson",
        ]

        experts = []
        for i in range(self.num_experts):
            experts.append(
                HumanExpert(
                    expert_id=i,
                    name=names[i % len(names)],
                    experience_years=random.randint(10, 25),
                    specializations=specializations[i % len(specializations)],
                    confidence_level=random.uniform(0.7, 0.95),
                )
            )
        return experts

    def _initialize_ai_system(self) -> AIOptimizer:
        """Set the attributes used to describe the simulated AI recommender."""
        return AIOptimizer(
            system_id=1,
            model_version=self.ai_system_version,
            optimization_capabilities=[
                "mathematical_optimization",
                "pattern_recognition",
                "predictive_analytics",
                "real_time_optimization",
                "explainable_ai",
            ],
            confidence_level=0.8,
            explainability_score=0.8,
        )

    def generate_vrp_instance(self, complexity: str = "medium") -> Dict:
        """Generate the location and demand data used in a routing instance."""
        problem_sizes = {
            "simple": (8, 2),
            "medium": (15, 3),
            "complex": (25, 4),
        }
        if complexity not in problem_sizes:
            raise ValueError("Complexity must be 'simple', 'medium', or 'complex'.")

        num_customers, num_vehicles = problem_sizes[complexity]
        depot = (0.0, 0.0)
        customers = []

        for i in range(num_customers):
            angle = 2 * np.pi * i / num_customers
            radius = np.random.uniform(5, 15)
            location = (radius * np.cos(angle), radius * np.sin(angle))
            customers.append(
                {
                    "id": i + 1,
                    "location": location,
                    "demand": int(np.random.randint(2, 8)),
                    "time_window": (8, 18),
                    "priority": random.choice(["low", "medium", "high"]),
                }
            )

        return {
            "depot": depot,
            "customers": customers,
            "vehicles": [
                {"id": i + 1, "capacity": 20} for i in range(num_vehicles)
            ],
            "complexity": complexity,
            "timestamp": datetime.now(),
        }

    def human_expert_decision(
        self, expert: HumanExpert, problem_instance: Dict
    ) -> List[List[int]]:
        """Build a complete route recommendation using an expert-style rule."""
        customers = problem_instance["customers"]
        num_vehicles = len(problem_instance["vehicles"])
        routes = [[] for _ in range(num_vehicles)]

        if "cost_optimization" in expert.specializations:
            # Nearest-neighbor assignment gives the cost-focused expert a
            # distance-based rule while distributing work across vehicles.
            remaining = customers.copy()
            vehicle_index = 0
            current_position = problem_instance["depot"]

            while remaining:
                nearest = min(
                    remaining,
                    key=lambda customer: self._euclidean_distance(
                        current_position, customer["location"]
                    ),
                )
                routes[vehicle_index].append(nearest["id"])
                current_position = nearest["location"]
                remaining.remove(nearest)

                if len(routes[vehicle_index]) >= 4 and remaining:
                    vehicle_index = (vehicle_index + 1) % num_vehicles
                    current_position = problem_instance["depot"]
        else:
            # Other experts use their domain profile to choose a simple
            # assignment order; every customer is assigned exactly once.
            ordered_customers = customers.copy()
            if "urban_routing" in expert.specializations:
                ordered_customers.sort(
                    key=lambda customer: np.arctan2(
                        customer["location"][1], customer["location"][0]
                    )
                )
            else:
                random.shuffle(ordered_customers)

            for index, customer in enumerate(ordered_customers):
                routes[index % num_vehicles].append(customer["id"])

        if expert.experience_years > 15:
            routes = [self._two_opt_route(route, problem_instance) for route in routes]

        self._validate_routes(routes, problem_instance)
        return routes

    def ai_optimization_decision(self, problem_instance: Dict) -> List[List[int]]:
        """Create an AI-style recommendation using clustering and local search."""
        customers = problem_instance["customers"]
        num_vehicles = len(problem_instance["vehicles"])
        routes = [[] for _ in range(num_vehicles)]
        remaining = customers.copy()

        for vehicle_index in range(num_vehicles):
            if not remaining:
                break

            cluster_center = remaining[0]["location"]
            cluster = [remaining.pop(0)]

            for customer in remaining[:]:
                if len(cluster) >= 5:
                    break

                distance = self._euclidean_distance(
                    customer["location"], cluster_center
                )
                probability = np.exp(-distance / 10) * 0.8
                if random.random() < probability:
                    cluster.append(customer)
                    remaining.remove(customer)

            cluster_route = [customer["id"] for customer in cluster]
            routes[vehicle_index] = self._two_opt_route(
                cluster_route, problem_instance, max_passes=3
            )

        # Assign any customers not selected into a cluster to the route with
        # the fewest stops, so the recommendation remains a full partition.
        for customer in remaining:
            least_loaded = min(range(num_vehicles), key=lambda i: len(routes[i]))
            routes[least_loaded].append(customer["id"])

        self._validate_routes(routes, problem_instance)
        return routes

    @staticmethod
    def _euclidean_distance(point_a: Tuple[float, float], point_b: Tuple[float, float]) -> float:
        """Measure straight-line distance in the generated coordinate plane."""
        return float(np.hypot(point_a[0] - point_b[0], point_a[1] - point_b[1]))

    def _route_distance(self, route: List[int], problem_instance: Dict) -> float:
        """Compute one route's distance, including both depot legs."""
        if not route:
            return 0.0

        customers = {customer["id"]: customer for customer in problem_instance["customers"]}
        depot = problem_instance["depot"]
        distance = 0.0
        current_position = depot

        for customer_id in route:
            next_position = customers[customer_id]["location"]
            distance += self._euclidean_distance(current_position, next_position)
            current_position = next_position

        return distance + self._euclidean_distance(current_position, depot)

    def _two_opt_route(
        self,
        route: List[int],
        problem_instance: Dict,
        max_passes: int = 5,
    ) -> List[int]:
        """Improve a route by accepting distance-reducing segment reversals."""
        best_route = route.copy()
        if len(best_route) < 3:
            return best_route

        for _ in range(max_passes):
            improved = False
            for start in range(len(best_route) - 1):
                for end in range(start + 1, len(best_route)):
                    candidate = best_route.copy()
                    candidate[start : end + 1] = reversed(
                        candidate[start : end + 1]
                    )
                    if self._route_distance(candidate, problem_instance) < self._route_distance(
                        best_route, problem_instance
                    ):
                        best_route = candidate
                        improved = True
            if not improved:
                break
        return best_route

    @staticmethod
    def _validate_routes(routes: List[List[int]], problem_instance: Dict) -> None:
        """Check route count and exact customer coverage for a recommendation."""
        expected_ids = [customer["id"] for customer in problem_instance["customers"]]
        assigned_ids = [customer_id for route in routes for customer_id in route]
        assert len(routes) == len(problem_instance["vehicles"])
        assert sorted(assigned_ids) == sorted(expected_ids)

    def _calculate_route_cost(
        self, routes: List[List[int]], problem_instance: Dict
    ) -> float:
        """Sum depot-to-depot Euclidean distances over all vehicle routes."""
        self._validate_routes(routes, problem_instance)
        return sum(self._route_distance(route, problem_instance) for route in routes)

    def _generate_human_reasoning(self, expert: HumanExpert) -> str:
        """Summarize the profile-based rule behind an expert recommendation."""
        parts = [f"Experience: {expert.experience_years} years"]
        parts.append(f"Specialization: {', '.join(expert.specializations)}")
        if "urban_routing" in expert.specializations:
            parts.append("Used an angular ordering of customer locations")
        elif "cost_optimization" in expert.specializations:
            parts.append("Used nearest-neighbor assignment")
        if expert.experience_years > 15:
            parts.append("Applied route-level 2-opt local search")
        return "; ".join(parts)

    def _generate_ai_reasoning(self, route: List[List[int]]) -> str:
        """Summarize the clustering and local-search steps used by the AI heuristic."""
        customer_count = sum(len(vehicle_route) for vehicle_route in route)
        return (
            f"Model version: {self.ai_optimizer.model_version}; "
            "used probabilistic clustering and route-level 2-opt; "
            f"assigned {customer_count} customers across {len(route)} routes; "
            f"explainability attribute: {self.ai_optimizer.explainability_score:.2f}"
        )

    def _collaborative_refinement(
        self,
        human_recommendations: List[Dict],
        ai_recommendation: Dict,
        problem_instance: Dict,
    ) -> List[List[int]]:
        """Select a complete recommendation and refine each route with 2-opt."""
        collaborative_route = [route.copy() for route in ai_recommendation["route"]]
        ai_cost = self._calculate_route_cost(collaborative_route, problem_instance)

        if human_recommendations:
            best_human = min(
                human_recommendations,
                key=lambda recommendation: self._calculate_route_cost(
                    recommendation["route"], problem_instance
                ),
            )
            best_human_cost = self._calculate_route_cost(
                best_human["route"], problem_instance
            )
            if best_human_cost < 0.9 * ai_cost:
                collaborative_route = [route.copy() for route in best_human["route"]]

        collaborative_route = [
            self._two_opt_route(route, problem_instance)
            for route in collaborative_route
        ]
        self._validate_routes(collaborative_route, problem_instance)
        return collaborative_route

    def _calculate_final_confidence(
        self,
        human_confidence: float,
        ai_confidence: float,
        human_costs: List[float],
        ai_cost: float,
    ) -> float:
        """Combine assigned confidence, cost agreement, and current trust."""
        base_confidence = (human_confidence + ai_confidence) / 2
        if human_costs:
            average_human_cost = float(np.mean(human_costs))
            largest_cost = max(average_human_cost, ai_cost)
            cost_agreement = (
                1.0 - abs(average_human_cost - ai_cost) / largest_cost
                if largest_cost > 0
                else 1.0
            )
        else:
            cost_agreement = 0.5

        confidence = (
            0.6 * base_confidence
            + 0.2 * cost_agreement
            + 0.2 * self.trust_metrics.mutual_trust
        )
        return float(np.clip(confidence, 0.0, 1.0))

    @staticmethod
    def _calculate_agreement_level(human_costs: List[float], ai_cost: float) -> float:
        """Measure similarity of average human and AI costs, not route identity."""
        if not human_costs:
            return 0.5
        average_human_cost = float(np.mean(human_costs))
        largest_cost = max(average_human_cost, ai_cost)
        if largest_cost == 0:
            return 1.0
        return float(1.0 - abs(average_human_cost - ai_cost) / largest_cost)

    def collaborative_decision_making(self, problem_instance: Dict) -> VRPDecision:
        """Generate recommendations, refine a route, and update trust metrics."""
        decision_start = time.perf_counter()
        human_recommendations = []

        for expert in self.human_experts:
            route = self.human_expert_decision(expert, problem_instance)
            human_recommendations.append(
                {
                    "expert_id": expert.expert_id,
                    "expert_name": expert.name,
                    "route": route,
                    "confidence": expert.confidence_level,
                    "reasoning": self._generate_human_reasoning(expert),
                }
            )

        ai_route = self.ai_optimization_decision(problem_instance)
        ai_recommendation = {
            "system_id": self.ai_optimizer.system_id,
            "route": ai_route,
            "confidence": self.ai_optimizer.confidence_level,
            "reasoning": self._generate_ai_reasoning(ai_route),
            "optimization_method": "probabilistic clustering and 2-opt local search",
        }

        human_costs = [
            self._calculate_route_cost(item["route"], problem_instance)
            for item in human_recommendations
        ]
        ai_cost = self._calculate_route_cost(ai_route, problem_instance)
        collaborative_route = self._collaborative_refinement(
            human_recommendations, ai_recommendation, problem_instance
        )
        average_human_confidence = float(
            np.mean([item["confidence"] for item in human_recommendations])
        )
        final_confidence = self._calculate_final_confidence(
            average_human_confidence,
            ai_recommendation["confidence"],
            human_costs,
            ai_cost,
        )
        decision_time = time.perf_counter() - decision_start

        decision = VRPDecision(
            decision_id=len(self.decision_history) + 1,
            timestamp=datetime.now(),
            problem_instance=problem_instance,
            human_recommendation=human_recommendations[0]["route"],
            ai_recommendation=ai_route,
            collaborative_decision=collaborative_route,
            human_confidence=average_human_confidence,
            ai_confidence=ai_recommendation["confidence"],
            final_confidence=final_confidence,
            outcome={
                "decision_time": decision_time,
                "human_costs": human_costs,
                "ai_cost": ai_cost,
                "collaborative_cost": self._calculate_route_cost(
                    collaborative_route, problem_instance
                ),
                "agreement_level": self._calculate_agreement_level(
                    human_costs, ai_cost
                ),
                "trust_at_decision": self.trust_metrics.mutual_trust,
            },
        )

        self._update_trust_metrics(decision)
        self.decision_history.append(decision)
        self.collaboration_count += 1
        self.performance_metrics["human_only_cost"].append(float(np.mean(human_costs)))
        self.performance_metrics["ai_only_cost"].append(ai_cost)
        self.performance_metrics["collaborative_cost"].append(
            decision.outcome["collaborative_cost"]
        )
        self.performance_metrics["decision_time"].append(decision_time)
        return decision

    def _update_trust_metrics(self, decision: VRPDecision) -> None:
        """Apply the stated bounded trust update rule to one decision."""
        human_costs = decision.outcome["human_costs"]
        if human_costs:
            average_human_cost = float(np.mean(human_costs))
            collaborative_cost = decision.outcome["collaborative_cost"]
            if average_human_cost > 0:
                improvement = (average_human_cost - collaborative_cost) / average_human_cost
                if improvement > 0.1:
                    self.trust_metrics.human_trust_in_ai = min(
                        1.0, self.trust_metrics.human_trust_in_ai + 0.05
                    )
                    self.trust_metrics.ai_trust_in_human = min(
                        1.0, self.trust_metrics.ai_trust_in_human + 0.03
                    )
                elif improvement < -0.05:
                    self.trust_metrics.human_trust_in_ai = max(
                        0.0, self.trust_metrics.human_trust_in_ai - 0.02
                    )
                    self.trust_metrics.ai_trust_in_human = max(
                        0.0, self.trust_metrics.ai_trust_in_human - 0.01
                    )

        self.trust_metrics.mutual_trust = (
            self.trust_metrics.human_trust_in_ai
            + self.trust_metrics.ai_trust_in_human
        ) / 2
        self.trust_metrics.trust_history.append(
            (decision.timestamp, self.trust_metrics.mutual_trust)
        )
        self.trust_metrics.trust_factors["accuracy"] = decision.final_confidence
        self.trust_metrics.trust_factors["reliability"] = decision.outcome[
            "agreement_level"
        ]
        self.trust_metrics.trust_factors["responsiveness"] = 1.0 / (
            1.0 + decision.outcome["decision_time"]
        )

        if self.trust_metrics.mutual_trust > 0.8:
            self.partnership_stage = "performing"
        elif self.trust_metrics.mutual_trust > 0.6:
            self.partnership_stage = "norming"
        elif self.trust_metrics.mutual_trust > 0.4:
            self.partnership_stage = "storming"
        else:
            self.partnership_stage = "forming"

In [ ]:
# Initialize the simulated human–AI partnership.
print("=== INITIALIZING THE HUMAN–AI PARTNERSHIP ===")
symbiotic_system = HumanAISymbioticSystem(num_experts=5)
print(f"Human experts: {symbiotic_system.num_experts}")
print(f"AI system version: {symbiotic_system.ai_optimizer.model_version}")
print(f"Initial mutual trust: {symbiotic_system.trust_metrics.mutual_trust:.2f}")
print()

print("=== HUMAN EXPERT PROFILES ===")
for expert in symbiotic_system.human_experts:
    print(f"{expert.name} (expert {expert.expert_id}):")
    print(f"  Experience: {expert.experience_years} years")
    print(f"  Specializations: {', '.join(expert.specializations)}")
    print(f"  Assigned confidence attribute: {expert.confidence_level:.2f}")
    print(f"  Initial trust in AI: {expert.trust_in_ai:.2f}")

print("=== AI SYSTEM ATTRIBUTES ===")
print(f"Capabilities: {', '.join(symbiotic_system.ai_optimizer.optimization_capabilities)}")
print(f"Assigned confidence attribute: {symbiotic_system.ai_optimizer.confidence_level:.2f}")
print(f"Explainability attribute: {symbiotic_system.ai_optimizer.explainability_score:.2f}")

simple_problem = symbiotic_system.generate_vrp_instance("simple")
medium_problem = symbiotic_system.generate_vrp_instance("medium")
complex_problem = symbiotic_system.generate_vrp_instance("complex")
problems = [
    ("Simple", simple_problem),
    ("Medium", medium_problem),
    ("Complex", complex_problem),
]

print("=== GENERATED PROBLEM SIZES ===")
for name, problem in problems:
    print(
        f"{name}: {len(problem['customers'])} customers, "
        f"{len(problem['vehicles'])} vehicles"
    )

In [ ]:
print("=== COLLABORATIVE ROUTING DECISIONS ===")
all_decisions = []

for complexity, problem in problems:
    decision = symbiotic_system.collaborative_decision_making(problem)
    all_decisions.append(decision)

    print(f"{complexity} problem — decision {decision.decision_id}")
    print(f"  Decision time: {decision.outcome['decision_time']:.4f} seconds")
    print(f"  Mean human confidence attribute: {decision.human_confidence:.2f}")
    print(f"  AI confidence attribute: {decision.ai_confidence:.2f}")
    print(f"  Combined confidence measure: {decision.final_confidence:.2f}")
    print(f"  Cost agreement measure: {decision.outcome['agreement_level']:.2f}")
    print(f"  Mean human route cost: {np.mean(decision.outcome['human_costs']):.2f}")
    print(f"  AI route cost: {decision.outcome['ai_cost']:.2f}")
    print(f"  Collaborative route cost: {decision.outcome['collaborative_cost']:.2f}")
    print(f"  Mutual trust at decision time: {decision.outcome['trust_at_decision']:.2f}")
    print(f"  Partnership stage after update: {symbiotic_system.partnership_stage}")

print()
print("=== REPEATED DECISIONS ===")
print(
    "Ten additional decisions illustrate the specified trust update rule; "
    "they do not train or update the AI routing policy."
)

for _ in range(10):
    complexity = random.choice(["simple", "medium", "complex"])
    problem = symbiotic_system.generate_vrp_instance(complexity)
    all_decisions.append(
        symbiotic_system.collaborative_decision_making(problem)
    )

print(f"Total simulated decisions: {len(all_decisions)}")
print(f"Final mutual trust measure: {symbiotic_system.trust_metrics.mutual_trust:.2f}")
print(f"Final partnership stage: {symbiotic_system.partnership_stage}")

In [ ]:
print("=== PARTNERSHIP AND ROUTE-COST SUMMARY ===")

trust_history = symbiotic_system.trust_metrics.trust_history
if trust_history:
    initial_trust = trust_history[0][1]
    final_trust = trust_history[-1][1]
    trust_change = final_trust - initial_trust
    print(f"Trust measure at first recorded decision: {initial_trust:.2f}")
    print(f"Trust measure at last recorded decision: {final_trust:.2f}")
    print(f"Change in trust measure: {trust_change:+.2f}")
    print("Trust-factor values:")
    for factor, value in symbiotic_system.trust_metrics.trust_factors.items():
        print(f"  {factor.replace('_', ' ').title()}: {value:.2f}")

print()
print("=== CONFIDENCE AND COST INDICATORS ===")
human_confidence_values = [decision.human_confidence for decision in all_decisions]
ai_confidence_values = [decision.ai_confidence for decision in all_decisions]
collaborative_confidence_values = [
    decision.final_confidence for decision in all_decisions
]
print(
    "Mean human confidence attribute: "
    f"{np.mean(human_confidence_values):.2%}"
)
print(f"Mean AI confidence attribute: {np.mean(ai_confidence_values):.2%}")
print(
    "Mean combined confidence measure: "
    f"{np.mean(collaborative_confidence_values):.2%}"
)
print(
    "These are simulation attributes and derived measures, not observed "
    "decision-accuracy rates."
)

cost_reductions = []
for decision in all_decisions:
    average_human_cost = float(np.mean(decision.outcome["human_costs"]))
    collaborative_cost = decision.outcome["collaborative_cost"]
    if average_human_cost > 0:
        cost_reductions.append(
            (average_human_cost - collaborative_cost) / average_human_cost
        )

if cost_reductions:
    print(
        "Mean relative cost change versus the mean human recommendation: "
        f"{np.mean(cost_reductions) * 100:.2f}%"
    )
    print("Positive values indicate lower collaborative route cost; negative values indicate higher cost.")

decision_times = [decision.outcome["decision_time"] for decision in all_decisions]
print(f"Mean measured decision time: {np.mean(decision_times):.4f} seconds")
print("Execution time depends on the computing environment and is not a real-world service-level result.")

print()
print("=== INTERPRETATION ===")
print(
    "The cost comparison describes the generated instances and the heuristics "
    "implemented here. It does not establish global optimality, VRP feasibility "
    "under capacity or time-window constraints, or a causal benefit from human–AI collaboration."
)

In [ ]:
def visualize_symbiotic_partnership(
    symbiotic_system: HumanAISymbioticSystem,
    all_decisions: List[VRPDecision],
) -> None:
    """Plot trust, confidence measures, route-cost changes, and expert profiles."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    trust_axis, confidence_axis, cost_axis, specialization_axis = axes.flat

    trust_history = symbiotic_system.trust_metrics.trust_history
    if trust_history:
        decision_numbers = np.arange(1, len(trust_history) + 1)
        trust_values = [value for _, value in trust_history]
        trust_axis.plot(
            decision_numbers,
            trust_values,
            color="navy",
            marker="o",
            linewidth=2,
            label="Simulated mutual trust",
        )
        trust_axis.set_xlabel("Decision number")
        trust_axis.set_ylabel("Trust measure (0–1)")
        trust_axis.set_title("Rule-Based Trust Measure")
        trust_axis.set_ylim(0, 1)
        trust_axis.legend()
        trust_axis.grid(True, alpha=0.3)

    if all_decisions:
        decision_numbers = np.arange(1, len(all_decisions) + 1)
        confidence_axis.plot(
            decision_numbers,
            [item.human_confidence for item in all_decisions],
            marker="o",
            label="Mean human confidence attribute",
        )
        confidence_axis.plot(
            decision_numbers,
            [item.ai_confidence for item in all_decisions],
            marker="s",
            label="AI confidence attribute",
        )
        confidence_axis.plot(
            decision_numbers,
            [item.final_confidence for item in all_decisions],
            marker="^",
            label="Combined confidence measure",
        )
        confidence_axis.set_xlabel("Decision number")
        confidence_axis.set_ylabel("Confidence measure (0–1)")
        confidence_axis.set_title("Assigned and Derived Confidence Measures")
        confidence_axis.set_ylim(0, 1)
        confidence_axis.legend(fontsize=8)
        confidence_axis.grid(True, alpha=0.3)

        relative_cost_changes = []
        for decision in all_decisions:
            mean_human_cost = float(np.mean(decision.outcome["human_costs"]))
            if mean_human_cost > 0:
                relative_cost_changes.append(
                    100
                    * (mean_human_cost - decision.outcome["collaborative_cost"])
                    / mean_human_cost
                )
        if relative_cost_changes:
            cost_axis.axhline(0, color="black", linewidth=1)
            cost_axis.plot(
                np.arange(1, len(relative_cost_changes) + 1),
                relative_cost_changes,
                color="darkgreen",
                marker="s",
                label="Relative change versus mean human cost",
            )
            cost_axis.set_xlabel("Decision number")
            cost_axis.set_ylabel("Cost reduction (%)")
            cost_axis.set_title("Collaborative Cost Comparison")
            cost_axis.legend(fontsize=8)
            cost_axis.grid(True, alpha=0.3)

    specialization_counts: Dict[str, int] = {}
    for expert in symbiotic_system.human_experts:
        for specialization in expert.specializations:
            specialization_counts[specialization] = (
                specialization_counts.get(specialization, 0) + 1
            )

    if specialization_counts:
        labels = list(specialization_counts)
        counts = [specialization_counts[label] for label in labels]
        bars = specialization_axis.bar(labels, counts, color="slateblue", alpha=0.75)
        specialization_axis.set_xlabel("Specialization")
        specialization_axis.set_ylabel("Number of experts")
        specialization_axis.set_title("Expert Specialization Distribution")
        specialization_axis.grid(True, axis="y", alpha=0.3)
        specialization_axis.tick_params(axis="x", labelrotation=45)
        for bar, count in zip(bars, counts):
            specialization_axis.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                str(count),
                ha="center",
                va="bottom",
            )

    fig.tight_layout()
    plt.show()


visualize_symbiotic_partnership(symbiotic_system, all_decisions)

## How Tier 7 relates to other solution tiers

The distinguishing feature of the human–AI partnership is the explicit interaction between human judgment and computational recommendations. The simulation makes that interaction visible through separate recommendations, a cost-based comparison, a refinement step, and a trust record. It does not implement every capability associated with a full human–AI operational platform.

- **Mathematical formulation:** a formal model defines decision variables, an objective, and constraints. This notebook uses a distance objective for comparisons but does not enforce capacity or time-window constraints and does not solve an exact mathematical program.
- **Simple heuristic:** a fixed rule generates a recommendation. The simulated experts use profile-dependent assignment rules, and the AI uses clustering-style assignment.
- **Metaheuristic:** a broader search procedure explores candidate solutions. The local 2-opt step here is a bounded route-order improvement heuristic, not a general metaheuristic framework.
- **Continual learning:** a system updates its predictive or decision policy using new data. The notebook stores feedback-related data structures but does not implement a learning update; repeated decisions only exercise the stated trust rule.
- **Digital twin:** a digital representation supports simulation or operational decisions. The generated coordinate instances are small synthetic examples, not a validated digital twin.
- **Multi-agent system:** autonomous agents coordinate through defined mechanisms. The human and AI components here are simulated recommenders coordinated by one central system.

## Strengths and limitations of the partnership framing

**Potential strengths in an operational design:**

- Human domain expertise can identify practical considerations missing from a simplified objective.
- Separate recommendations and recorded reasoning can support review and accountability.
- Comparing route costs makes one part of the refinement decision inspectable.
- Human oversight can be valuable when decisions have safety, service, or regulatory consequences.

**Limitations of this simulation:**

- The expert and AI policies are hand-coded demonstrations rather than validated decision systems.
- Confidence, trust, explainability, and responsiveness are simplified attributes or derived measures; they are not independently measured human factors.
- The route-cost objective omits the generated capacity, demand, time-window, traffic, service-time, and priority data.
- The collaboration rule selects among complete recommendations and applies route-level local search; it does not model a dialogue, structured disagreement process, or feedback-driven AI learning.
- The synthetic results cannot substantiate real-world accuracy, cost savings, decision speed, stakeholder satisfaction, or causal improvement.

## Contextual figures and interpretation

The motivating example associated with this topic reports a regional logistics partnership with five human experts and an AI system, including reported trust, decision-quality, cost, speed, conflict-resolution, satisfaction, and learning figures. Those values describe that example; they are not outputs of this synthetic notebook, and the simulation does not test whether they are achievable. Any such figures should be interpreted with the source study's measurement definitions, experimental design, and operational context.

## When this tier may be useful

A human-in-the-loop design may be worth considering when route decisions require contextual judgment, when the consequences of a recommendation are significant, or when operators need a reviewable basis for accepting or rejecting recommendations. A production system would need explicit operational constraints, validated data and performance measures, documented decision authority, and a tested process for handling disagreement. Those additions are outside the implementation in this notebook.